In [ ]:
import cv2
import numpy as np
import time
from collections import deque

import ipywidgets as widgets
import traitlets
from IPython.display import display

from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg

# Nếu đã có camera từ notebook khác, không tạo lại nhiều lần
try:
    camera.running = False
except:
    pass

camera = CSICamera(width=224, height=224)
camera.running = True

camera_widget = widgets.Image(format='jpeg', width=224, height=224)

camera_link = traitlets.dlink(
    (camera, 'value'),
    (camera_widget, 'value'),
    transform=bgr8_to_jpeg
)

display(camera_widget)

print("Camera started")

In [ ]:
class LaneDetector:
    def __init__(self):
        self.roi_start_ratio = 0.35

        # ROI hình thang, chỉnh theo camera
        self.poly_top_left = 35
        self.poly_top_right = 190
        self.poly_bottom_left = 5
        self.poly_bottom_right = 219

        # Threshold vạch trắng
        # Nếu bắt thiếu vạch: giảm V từ 145 xuống 125
        # Nếu bắt quá nhiều nền trắng: tăng V lên 165 và giảm S upper
        self.lower_white = np.array([0, 0, 145])
        self.upper_white = np.array([180, 80, 255])

        # Threshold mặt đường đen để loại nền trắng ngoài đường
        self.lower_black = np.array([0, 0, 0])
        self.upper_black = np.array([180, 255, 115])

        # Lane width fallback nếu chỉ thấy một vạch
        self.lane_width_pixels = 90

        # Chọn lane:
        # "RIGHT_LANE": chạy giữa vạch đứt bên trái và vạch liền bên phải
        # "LEFT_LANE": chạy giữa vạch liền bên trái và vạch đứt bên phải
        self.target_lane = "RIGHT_LANE"

        self.lookahead_ratio = 0.75
        self.error_history = deque(maxlen=2)

        # Geometry/control calibration. Original Canny notebook values.
        self.geom_far_ratio = 0.35
        self.geom_near_ratio = 0.82
        self.target_offset_px = -18
        self.right_lane_bias_px = 0
        self.left_lane_bias_px = 0
        self.single_line_extra_offset_px = 30
        self.min_right_clearance_px = 48
        self.min_left_clearance_px = 34

        # Learn lane width when both boundaries are visible, then use it for one-line fallback.
        self.auto_lane_width = True
        self.lane_width_min_px = 55
        self.lane_width_max_px = 120
        self.lane_width_alpha = 0.18

        self.last_center = None
        self.lost_count = 0
        self.max_lost_count = 5

        self.color_mode = "BGR"

    def apply_roi(self, mask):
        h, w = mask.shape[:2]
        roi_mask = np.zeros_like(mask)

        polygon = np.array([[
            (self.poly_top_left, 0),
            (self.poly_top_right, 0),
            (self.poly_bottom_right, h - 1),
            (self.poly_bottom_left, h - 1)
        ]], dtype=np.int32)

        cv2.fillPoly(roi_mask, polygon, 255)
        return cv2.bitwise_and(mask, roi_mask)

    def fit_x_from_y(self, points):
        if len(points) < 6:
            return None

        xs = np.array([p[0] for p in points], dtype=np.float32)
        ys = np.array([p[1] for p in points], dtype=np.float32)

        a, b = np.polyfit(ys, xs, 1)
        return a, b

    def x_at_y(self, line, y):
        if line is None:
            return None
        a, b = line
        return int(a * y + b)

    def get_white_lane_mask(self, roi):
        if self.color_mode == "BGR":
            hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        else:
            hsv = cv2.cvtColor(roi, cv2.COLOR_RGB2HSV)

        white_mask = cv2.inRange(hsv, self.lower_white, self.upper_white)
        black_mask = cv2.inRange(hsv, self.lower_black, self.upper_black)

        # Chỉ xét trong ROI hình thang
        white_mask = self.apply_roi(white_mask)
        black_mask = self.apply_roi(black_mask)

        # Mở rộng vùng mặt đường đen một chút để giữ vạch trắng nằm sát mặt đường
        road_kernel = np.ones((17, 17), np.uint8)
        road_area = cv2.dilate(black_mask, road_kernel, iterations=1)

        # Chỉ giữ vạch trắng nằm gần/thuộc vùng mặt đường
        lane_mask = cv2.bitwise_and(white_mask, road_area)

        # Làm sạch nhiễu nhỏ
        lane_mask = cv2.morphologyEx(lane_mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

        # Nối nhẹ vạch đứt theo chiều dọc
        lane_mask = cv2.morphologyEx(lane_mask, cv2.MORPH_CLOSE, np.ones((13, 3), np.uint8))

        return lane_mask

    def extract_lane_lines(self, lane_mask):
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(lane_mask, connectivity=8)

        lines = []

        for i in range(1, num_labels):
            x = stats[i, cv2.CC_STAT_LEFT]
            y = stats[i, cv2.CC_STAT_TOP]
            w = stats[i, cv2.CC_STAT_WIDTH]
            h = stats[i, cv2.CC_STAT_HEIGHT]
            area = stats[i, cv2.CC_STAT_AREA]

            # Bỏ nhiễu quá nhỏ
            if area < 20:
                continue

            # Bỏ mảng trắng quá rộng, thường là nền/tường
            if w > 55 and h < 35:
                continue

            # Lấy points của component
            ys, xs = np.where(labels == i)

            points = list(zip(xs, ys))
            line = self.fit_x_from_y(points)

            if line is None:
                continue

            # Vị trí x tại lookahead để phân loại trái/phải
            lookahead_y = int(lane_mask.shape[0] * self.lookahead_ratio)
            x_at_lookahead = self.x_at_y(line, lookahead_y)

            if x_at_lookahead is None:
                continue

            lines.append({
                "line": line,
                "x": x_at_lookahead,
                "area": area,
                "bbox": (x, y, w, h)
            })

        lines = sorted(lines, key=lambda item: item["x"])
        return lines

    def select_lane_boundaries(self, lines, image_center):
        """
        Chọn 2 vạch trắng làm biên lane.
        RIGHT_LANE:
            left boundary = vạch trắng gần bên trái center
            right boundary = vạch trắng bên phải
        LEFT_LANE:
            left boundary = vạch bên trái
            right boundary = vạch gần bên phải center
        """
        if len(lines) < 1:
            return None, None

        left_lines = [ln for ln in lines if ln["x"] < image_center]
        right_lines = [ln for ln in lines if ln["x"] >= image_center]

        left_boundary = None
        right_boundary = None

        if self.target_lane == "RIGHT_LANE":
            # Vạch đứt thường nằm gần center/trái, vạch liền nằm phải
            if len(left_lines) > 0:
                left_boundary = max(left_lines, key=lambda ln: ln["x"])
            if len(right_lines) > 0:
                right_boundary = min(right_lines, key=lambda ln: ln["x"])

        else:
            # LEFT_LANE
            if len(left_lines) > 0:
                left_boundary = max(left_lines, key=lambda ln: ln["x"])
            if len(right_lines) > 0:
                right_boundary = min(right_lines, key=lambda ln: ln["x"])

        left_line = None if left_boundary is None else left_boundary["line"]
        right_line = None if right_boundary is None else right_boundary["line"]

        return left_line, right_line

    def detect(self, image):
        h, w = image.shape[:2]

        roi_y = int(h * self.roi_start_ratio)
        roi = image[roi_y:h, :]
        roi_h = roi.shape[0]

        lane_mask = self.get_white_lane_mask(roi)
        lines = self.extract_lane_lines(lane_mask)

        image_center = w // 2
        lookahead_y = int(roi_h * self.lookahead_ratio)

        left_line, right_line = self.select_lane_boundaries(lines, image_center)

        left_x = self.x_at_y(left_line, lookahead_y)
        right_x = self.x_at_y(right_line, lookahead_y)

        lane_center = None
        center_line = None

        if left_line is not None and right_line is not None:
            left_x = self.x_at_y(left_line, lookahead_y)
            right_x = self.x_at_y(right_line, lookahead_y)

            observed_width = right_x - left_x
            if self.auto_lane_width and self.lane_width_min_px <= observed_width <= self.lane_width_max_px:
                self.lane_width_pixels = int(
                    (1.0 - self.lane_width_alpha) * self.lane_width_pixels
                    + self.lane_width_alpha * observed_width
                )

            lane_center = int((left_x + right_x) / 2)

            a1, b1 = left_line
            a2, b2 = right_line
            center_line = ((a1 + a2) / 2, (b1 + b2) / 2)

        elif left_line is not None:
            left_x = self.x_at_y(left_line, lookahead_y)
            lane_center = int(left_x + self.lane_width_pixels / 2)

            a, b = left_line
            center_line = (a, b + self.lane_width_pixels / 2)

        elif right_line is not None:
            right_x = self.x_at_y(right_line, lookahead_y)
            lane_center = int(right_x - self.lane_width_pixels / 2)

            a, b = right_line
            center_line = (a, b - self.lane_width_pixels / 2)

        else:
            if self.last_center is not None and self.lost_count < self.max_lost_count:
                lane_center = self.last_center
                self.lost_count += 1
            else:
                return {
                    "ok": False,
                    "error": None,
                    "lane_center": None,
                    "left_x": None,
                    "right_x": None,
                    "mask": lane_mask,
                    "roi_y": roi_y,
                    "left_line": None,
                    "right_line": None,
                    "center_line": None,
                    "lookahead_y": lookahead_y,
                    "lines": lines
                }

        lane_center = max(0, min(w - 1, lane_center))

        self.last_center = lane_center
        self.lost_count = 0

        error = lane_center - image_center

        self.error_history.append(error)
        smooth_error = sum(self.error_history) / len(self.error_history)

        return {
            "ok": True,
            "error": smooth_error,
            "lane_center": lane_center,
            "left_x": left_x,
            "right_x": right_x,
            "mask": lane_mask,
            "roi_y": roi_y,
            "left_line": left_line,
            "right_line": right_line,
            "center_line": center_line,
            "lookahead_y": lookahead_y,
            "lines": lines
        }


detector = LaneDetector()
print("White lane marking detector ready")

In [ ]:
import math
import time

def get_road_geometry(result, detector, image):
    """
    Tính geometry điều khiển:
    - angle_deg: góc lệch giữa hướng lane và trục camera
    - lateral_error: target lane lệch so với camera center

    Bản này có thêm:
    - TARGET_OFFSET_PX để né line viền
    - SINGLE_LINE_EXTRA_OFFSET_PX để không bám 1 vạch duy nhất
    - border guard để target không nằm quá sát line viền
    """

    h, w = image.shape[:2]

    roi_y = result.get("roi_y", int(h * detector.roi_start_ratio))
    roi_h = h - roi_y

    center_line = result.get("center_line", None)

    # Điểm xa/gần để tính góc
    y_far = int(roi_h * getattr(detector, "geom_far_ratio", 0.35))
    y_near = int(roi_h * getattr(detector, "geom_near_ratio", 0.82))

    if center_line is not None:
        a, b = center_line
        x_far = int(a * y_far + b)
        x_near = int(a * y_near + b)
    else:
        x_near = result.get("lane_center", w // 2)
        if x_near is None:
            x_near = w // 2
        x_far = x_near

    left_x = result.get("left_x")
    right_x = result.get("right_x")

    # ==================================================
    # CHỈNH Ở ĐÂY
    # Nếu xe bám line viền phải: dùng số âm.
    # Nếu xe bám line viền trái: đổi thành số dương.
    # ==================================================
    TARGET_OFFSET_PX = getattr(detector, "target_offset_px", -18)
    if getattr(detector, "target_lane", "RIGHT_LANE") == "RIGHT_LANE":
        TARGET_OFFSET_PX += getattr(detector, "right_lane_bias_px", 0)
    else:
        TARGET_OFFSET_PX += getattr(detector, "left_lane_bias_px", 0)

    # Khi chỉ thấy 1 vạch, đẩy target ra xa vạch đó hơn
    SINGLE_LINE_EXTRA_OFFSET_PX = getattr(detector, "single_line_extra_offset_px", 30)

    # Không cho target quá sát line viền phải/trái
    MIN_RIGHT_CLEARANCE_PX = getattr(detector, "min_right_clearance_px", 48)
    MIN_LEFT_CLEARANCE_PX = getattr(detector, "min_left_clearance_px", 34)

    border_risk = False

    # Offset mặc định để né viền phải
    x_near += TARGET_OFFSET_PX
    x_far += int(TARGET_OFFSET_PX * 0.7)

    # Nếu chỉ thấy line phải, rất dễ bám line viền phải -> đẩy target sang trái thêm
    if left_x is None and right_x is not None:
        x_near -= SINGLE_LINE_EXTRA_OFFSET_PX
        x_far -= int(SINGLE_LINE_EXTRA_OFFSET_PX * 0.7)
        border_risk = True

    # Nếu chỉ thấy line trái -> đẩy target sang phải thêm
    elif right_x is None and left_x is not None:
        x_near += SINGLE_LINE_EXTRA_OFFSET_PX
        x_far += int(SINGLE_LINE_EXTRA_OFFSET_PX * 0.7)
        border_risk = True

    # Guard line phải: nếu target gần line phải quá thì kéo target sang trái
    if right_x is not None:
        dist_right = right_x - x_near

        if 0 < dist_right < MIN_RIGHT_CLEARANCE_PX:
            shift = MIN_RIGHT_CLEARANCE_PX - dist_right
            x_near -= shift
            x_far -= int(shift * 0.7)
            border_risk = True

    # Guard line trái: nếu target gần line trái quá thì kéo target sang phải
    if left_x is not None:
        dist_left = x_near - left_x

        if 0 < dist_left < MIN_LEFT_CLEARANCE_PX:
            shift = MIN_LEFT_CLEARANCE_PX - dist_left
            x_near += shift
            x_far += int(shift * 0.7)
            border_risk = True

    # Giới hạn trong ảnh
    x_near = clamp(x_near, 0, w - 1)
    x_far = clamp(x_far, 0, w - 1)

    image_center = w // 2

    lateral_error = x_near - image_center

    dx = x_near - x_far
    dy = y_near - y_far

    angle_rad = math.atan2(dx, dy)
    angle_deg = math.degrees(angle_rad)

    return {
        "x_far": int(x_far),
        "y_far": int(roi_y + y_far),
        "x_near": int(x_near),
        "y_near": int(roi_y + y_near),
        "lateral_error": lateral_error,
        "angle_deg": angle_deg,
        "border_risk": border_risk
    }

In [ ]:
debug_widget = widgets.Image(format='jpeg', width=448, height=224)
status_widget = widgets.HTML(value="Debug status: idle")

display(debug_widget)
display(status_widget)

debug_running = False

def draw_fitted_line(debug, line, roi_y, roi_h, color, thickness=2):
    if line is None:
        return

    a, b = line

    y1_roi = int(roi_h * 0.20)
    y2_roi = roi_h - 1

    x1 = int(a * y1_roi + b)
    x2 = int(a * y2_roi + b)

    y1 = roi_y + y1_roi
    y2 = roi_y + y2_roi

    cv2.line(debug, (x1, y1), (x2, y2), color, thickness)


def draw_fitted_line(debug, line, roi_y, roi_h, color, thickness=2):
    if line is None:
        return

    a, b = line

    y1_roi = int(roi_h * 0.20)
    y2_roi = roi_h - 1

    x1 = int(a * y1_roi + b)
    x2 = int(a * y2_roi + b)

    y1 = roi_y + y1_roi
    y2 = roi_y + y2_roi

    cv2.line(debug, (x1, y1), (x2, y2), color, thickness)


def draw_fitted_line(debug, line, roi_y, roi_h, color, thickness=2):
    if line is None:
        return

    a, b = line

    y1_roi = int(roi_h * 0.20)
    y2_roi = roi_h - 1

    x1 = int(a * y1_roi + b)
    x2 = int(a * y2_roi + b)

    y1 = roi_y + y1_roi
    y2 = roi_y + y2_roi

    cv2.line(debug, (x1, y1), (x2, y2), color, thickness)


def make_debug_image(image, result):
    h, w = image.shape[:2]
    debug = image.copy()

    roi_y = int(h * detector.roi_start_ratio)
    roi_h = h - roi_y

    # ROI hình thang
    polygon = np.array([[
        (detector.poly_top_left, roi_y),
        (detector.poly_top_right, roi_y),
        (detector.poly_bottom_right, h - 1),
        (detector.poly_bottom_left, h - 1)
    ]], dtype=np.int32)

    cv2.polylines(debug, polygon, True, (0, 165, 255), 2)

    # Trục giữa camera
    cv2.line(debug, (w // 2, 0), (w // 2, h), (255, 255, 0), 2)

    # Vẽ tất cả line detect được bằng màu xám nhạt
    for item in result.get("lines", []):
        draw_fitted_line(debug, item["line"], roi_y, roi_h, (180, 180, 180), 1)

    # Biên lane đã chọn
    draw_fitted_line(debug, result.get("left_line"), roi_y, roi_h, (255, 0, 0), 2)
    draw_fitted_line(debug, result.get("right_line"), roi_y, roi_h, (0, 255, 0), 2)

    # Center lane
    draw_fitted_line(debug, result.get("center_line"), roi_y, roi_h, (0, 0, 255), 3)

    # Điểm lookahead
    if result.get("lane_center") is not None:
        lookahead_y_img = roi_y + result["lookahead_y"]
        cv2.circle(debug, (result["lane_center"], lookahead_y_img), 5, (0, 0, 255), -1)
        cv2.line(debug, (0, lookahead_y_img), (w - 1, lookahead_y_img), (255, 255, 255), 1)

    # Hiện thông tin góc nếu có
    try:
        geom = get_road_geometry(result, detector, image)
        text = "angle: {:.1f} deg | lat: {:.1f}px".format(
            geom["angle_deg"],
            geom["lateral_error"]
        )

        cv2.putText(
            debug,
            text,
            (5, 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )
    except:
        pass

    # Mask bên phải
    mask = result["mask"]
    mask_color = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    mask_color = cv2.resize(mask_color, (w, h))

    combined = np.concatenate([debug, mask_color], axis=1)
    return combined

In [ ]:
def live_debug(duration_sec=10):
    start = time.time()
    
    while time.time() - start < duration_sec:
        image = camera.value
        result = detector.detect(image)
        debug_img = make_debug_image(image, result)
        debug_widget.value = bgr8_to_jpeg(debug_img)
        
        print(
            "ok:", result["ok"],
            "error:", None if result["error"] is None else round(result["error"], 2),
            "left:", result["left_x"],
            "right:", result["right_x"]
        )
        
        time.sleep(0.15)

# live_debug(duration_sec=10)  # uncomment to test camera debug

In [ ]:
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()
car.throttle = 0.0
car.steering = 0.0

print("Motor stopped")

In [ ]:
# ============================================================
# Topology Intersection Detector
# Detect ngã tư/ngã rẽ bằng vùng đường đen nối với phía trước xe.
# Không dùng Canny/Hough làm điều kiện chính vì dễ ăn vạch trắng, tường, bóng đổ.
# ============================================================

import cv2
import numpy as np
import time


class TopologyIntersectionDetector:
    """
    Detect intersection bằng road topology:
    1. Tạo mask mặt đường đen.
    2. Chỉ giữ connected component nối với vùng trước xe ở bottom-center.
    3. Tính điểm road ở 4 gate: center, front, left, right.
    4. Báo intersection khi road mở rộng ngang đủ lớn và có nhánh trái/phải.
    """

    def __init__(self):
        # Ngưỡng nhận mặt đường đen.
        # Nếu mask thiếu mặt đường: tăng v_max lên 125-140.
        # Nếu mask ăn vào tường/bóng/nhiễu: giảm v_max xuống 90-105.
        self.v_max = 115

        # Cắt bỏ phần trên ảnh để tránh tường/đèn/nền xa.
        self.ignore_top_ratio = 0.24

        # Gate thresholds.
        self.center_min = 0.14
        self.front_min = 0.12
        self.side_min = 0.16
        self.width_min = 0.55

        # Với ngã tư thật nên để True.
        # Nếu test ngã ba chữ T, có thể đổi thành False.
        self.require_front = False

        # Debounce/cooldown.
        self.stable_frames_enter = 2
        self.stable_frames_exit = 2
        self.cooldown_sec = 2.0
        self.stable_count = 0
        self.missing_count = 0
        self.cooldown_until = 0.0

        # Gate layout theo tỉ lệ ảnh.
        # Có thể chỉnh nếu camera nhìn quá gần/quá xa.
        self.rects = {
            "front":  (0.38, 0.28, 0.62, 0.50),
            "center": (0.34, 0.62, 0.66, 0.92),
            "left":   (0.04, 0.40, 0.34, 0.64),
            "right":  (0.66, 0.40, 0.96, 0.64),
        }

        # Band dùng để đo độ mở ngang của road.
        self.width_band = (0.42, 0.64)

        # Marking-topology guard. This is the main false-positive filter:
        # T-junction markings look like a horizontal T, 4-way markings like +.
        self.mark_ignore_top_ratio = 0.24
        self.mark_search_y = (0.28, 0.80)
        self.mark_center_x = (0.24, 0.76)
        self.horizontal_min_span = 0.28
        self.branch_min_span = 0.10
        self.vertical_min_span = 0.16
        self.plus_upper_min_span = 0.10
        self.mark_col_min = 0.12
        self.mark_row_min = 0.10
        self.require_stem_for_turn = False

    def reset(self):
        self.stable_count = 0
        self.missing_count = 0
        self.cooldown_until = 0.0

    def suppress(self):
        self.stable_count = 0
        self.missing_count = 0
        self.cooldown_until = time.time() + self.cooldown_sec

    def _dark_road_mask(self, image):
        h, w = image.shape[:2]

        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        v = hsv[:, :, 2]

        # Dùng V-channel để bắt mặt đường đen ổn định hơn Canny.
        mask = (v < self.v_max).astype(np.uint8) * 255

        # Nếu LaneDetector có threshold black thì AND thêm để đồng bộ với road following.
        try:
            detector_black = cv2.inRange(hsv, detector.lower_black, detector.upper_black)
            mask = cv2.bitwise_and(mask, detector_black)
        except Exception:
            pass

        # Bỏ vùng quá xa phía trên.
        mask[:int(h * self.ignore_top_ratio), :] = 0

        # Làm sạch mask.
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

        return mask

    def _connected_road_from_bottom_center(self, mask):
        """
        Chỉ giữ vùng road nối với khu vực trước xe.
        Cách này giúp loại bớt tường tối, bóng tối, hoặc vật thể không nối với đường xe đang chạy.
        """
        h, w = mask.shape[:2]
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)

        seed_y1 = int(h * 0.76)
        seed_y2 = int(h * 0.98)
        seed_x1 = int(w * 0.36)
        seed_x2 = int(w * 0.64)

        seed_labels = labels[seed_y1:seed_y2, seed_x1:seed_x2].reshape(-1)
        seed_labels = seed_labels[seed_labels > 0]

        if len(seed_labels) == 0:
            return np.zeros_like(mask)

        ids, counts = np.unique(seed_labels, return_counts=True)
        road_label = ids[np.argmax(counts)]
        road = (labels == road_label).astype(np.uint8) * 255
        return road

    def _score_rect(self, mask, rect):
        h, w = mask.shape[:2]
        x1r, y1r, x2r, y2r = rect
        x1 = int(w * x1r)
        y1 = int(h * y1r)
        x2 = int(w * x2r)
        y2 = int(h * y2r)

        roi = mask[y1:y2, x1:x2]
        if roi.size == 0:
            return 0.0
        return float(np.mean(roi > 0))

    def _road_width_fraction(self, road):
        h, w = road.shape[:2]
        y1r, y2r = self.width_band
        y1 = int(h * y1r)
        y2 = int(h * y2r)

        band = road[y1:y2, :] > 0
        if band.size == 0:
            return 0.0

        col_count = band.sum(axis=0)
        active_cols = np.where(col_count > 0.12 * band.shape[0])[0]
        if len(active_cols) == 0:
            return 0.0

        return float(active_cols[-1] - active_cols[0]) / float(w)

    def _white_marking_mask(self, image):
        h, w = image.shape[:2]
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        try:
            lower_white = detector.lower_white
            upper_white = detector.upper_white
        except Exception:
            lower_white = np.array([0, 0, 145])
            upper_white = np.array([180, 90, 255])

        mask = cv2.inRange(hsv, lower_white, upper_white)
        mask[:int(h * self.mark_ignore_top_ratio), :] = 0
        mask[int(h * 0.96):, :] = 0

        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
        clean = np.zeros_like(mask)
        for i in range(1, num_labels):
            x = stats[i, cv2.CC_STAT_LEFT]
            y = stats[i, cv2.CC_STAT_TOP]
            bw = stats[i, cv2.CC_STAT_WIDTH]
            bh = stats[i, cv2.CC_STAT_HEIGHT]
            area = stats[i, cv2.CC_STAT_AREA]

            if area < 10:
                continue
            if area > int(w * h * 0.10):
                continue
            if bw > int(w * 0.82) and bh > int(h * 0.18):
                continue

            clean[labels == i] = 255

        return clean

    def _runs_from_bool(self, values):
        runs = []
        start = None
        for i, v in enumerate(values):
            if v and start is None:
                start = i
            elif (not v) and start is not None:
                runs.append((start, i - 1))
                start = None
        if start is not None:
            runs.append((start, len(values) - 1))
        return runs

    def _longest_true_run_len(self, values):
        runs = self._runs_from_bool(values)
        if len(runs) == 0:
            return 0
        return max((b - a + 1) for a, b in runs)

    def _measure_marking_topology(self, marking):
        h, w = marking.shape[:2]

        horizontal_mask = cv2.morphologyEx(marking, cv2.MORPH_CLOSE, np.ones((5, 31), np.uint8))
        horizontal_mask = cv2.dilate(horizontal_mask, np.ones((3, 9), np.uint8), iterations=1)
        vertical_mask = cv2.morphologyEx(marking, cv2.MORPH_CLOSE, np.ones((31, 5), np.uint8))
        vertical_mask = cv2.dilate(vertical_mask, np.ones((9, 3), np.uint8), iterations=1)

        y1 = int(h * self.mark_search_y[0])
        y2 = int(h * self.mark_search_y[1])
        x_center = w // 2
        cx1 = int(w * self.mark_center_x[0])
        cx2 = int(w * self.mark_center_x[1])

        band = horizontal_mask[y1:y2, :] > 0
        if band.size == 0:
            return self._empty_marking_topology(marking, horizontal_mask, vertical_mask)

        row_score = np.mean(band, axis=1)
        if len(row_score) >= 7:
            row_score = np.convolve(row_score, np.ones(7) / 7.0, mode='same')

        best_row = int(np.argmax(row_score))
        bar_y = int(y1 + best_row)
        bar_score = float(row_score[best_row])

        strip_y1 = max(0, bar_y - 7)
        strip_y2 = min(h, bar_y + 8)
        bar_strip = horizontal_mask[strip_y1:strip_y2, :] > 0
        col_score = np.mean(bar_strip, axis=0)
        active_cols = col_score >= self.mark_col_min
        runs = self._runs_from_bool(active_cols)
        centered_runs = [r for r in runs if r[1] >= cx1 and r[0] <= cx2]

        if len(centered_runs) == 0:
            return self._empty_marking_topology(marking, horizontal_mask, vertical_mask, bar_y=bar_y, bar_score=bar_score)

        bar_run = max(centered_runs, key=lambda r: r[1] - r[0])
        bar_x1, bar_x2 = int(bar_run[0]), int(bar_run[1])
        bar_span = float(bar_x2 - bar_x1 + 1) / float(w)
        horizontal_ok = (bar_span >= self.horizontal_min_span) and (bar_score >= self.mark_row_min)

        stem_search_x1 = max(bar_x1, cx1)
        stem_search_x2 = min(bar_x2 + 1, cx2)
        stem_x = None
        down_span = 0.0
        up_span = 0.0
        stem_down_ok = False
        stem_up_ok = False

        if stem_search_x2 > stem_search_x1:
            below_y1 = bar_y
            below_y2 = int(h * 0.94)
            below = vertical_mask[below_y1:below_y2, stem_search_x1:stem_search_x2] > 0
            if below.size > 0:
                col_cover = np.mean(below, axis=0)
                best_col = int(np.argmax(col_cover))
                stem_x = int(stem_search_x1 + best_col)

                win = max(4, int(w * 0.035))
                sx1 = max(0, stem_x - win)
                sx2 = min(w, stem_x + win + 1)
                stem_below = vertical_mask[below_y1:below_y2, sx1:sx2] > 0
                active_down_rows = np.mean(stem_below, axis=1) >= self.mark_row_min
                down_span = float(self._longest_true_run_len(active_down_rows)) / float(h)
                stem_down_ok = down_span >= self.vertical_min_span

                above_y1 = int(h * self.mark_ignore_top_ratio)
                above_y2 = max(above_y1 + 1, bar_y)
                stem_above = vertical_mask[above_y1:above_y2, sx1:sx2] > 0
                active_up_rows = np.mean(stem_above, axis=1) >= self.mark_row_min
                up_span = float(self._longest_true_run_len(active_up_rows)) / float(h)
                stem_up_ok = up_span >= self.plus_upper_min_span

        if stem_x is None:
            stem_x = x_center

        left_span = float(stem_x - bar_x1) / float(w)
        right_span = float(bar_x2 - stem_x) / float(w)
        stem_guard_ok = (not self.require_stem_for_turn) or stem_down_ok
        branch_left = horizontal_ok and stem_guard_ok and (left_span >= self.branch_min_span)
        branch_right = horizontal_ok and stem_guard_ok and (right_span >= self.branch_min_span)
        straight = horizontal_ok and stem_up_ok

        if straight:
            shape_type = 'plus'
        elif branch_left or branch_right:
            shape_type = 't_junction'
        else:
            shape_type = 'none'

        return {
            'marking_mask': marking,
            'horizontal_mask': horizontal_mask,
            'vertical_mask': vertical_mask,
            'horizontal_ok': horizontal_ok,
            'stem_down_ok': stem_down_ok,
            'stem_up_ok': stem_up_ok,
            'branch_left': branch_left,
            'branch_right': branch_right,
            'straight': straight,
            'shape_type': shape_type,
            'bar_y': bar_y,
            'bar_run': (bar_x1, bar_x2),
            'stem_x': int(stem_x),
            'bar_span': bar_span,
            'left_span': left_span,
            'right_span': right_span,
            'down_span': down_span,
            'up_span': up_span,
            'bar_score': bar_score,
        }

    def _empty_marking_topology(self, marking, horizontal_mask, vertical_mask, bar_y=None, bar_score=0.0):
        return {
            'marking_mask': marking,
            'horizontal_mask': horizontal_mask,
            'vertical_mask': vertical_mask,
            'horizontal_ok': False,
            'stem_down_ok': False,
            'stem_up_ok': False,
            'branch_left': False,
            'branch_right': False,
            'straight': False,
            'shape_type': 'none',
            'bar_y': bar_y,
            'bar_run': None,
            'stem_x': None,
            'bar_span': 0.0,
            'left_span': 0.0,
            'right_span': 0.0,
            'down_span': 0.0,
            'up_span': 0.0,
            'bar_score': bar_score,
        }

    def detect(self, image):
        now = time.time()
        mask = self._dark_road_mask(image)
        road = self._connected_road_from_bottom_center(mask)
        marking_mask = self._white_marking_mask(image)
        marking_topology = self._measure_marking_topology(marking_mask)

        front_score = self._score_rect(road, self.rects["front"])
        center_score = self._score_rect(road, self.rects["center"])
        left_score = self._score_rect(road, self.rects["left"])
        right_score = self._score_rect(road, self.rects["right"])
        width_frac = self._road_width_fraction(road)

        front_ok = front_score >= self.front_min
        center_ok = center_score >= self.center_min
        # Detect nhánh trái/phải bằng "side exit", không chỉ bằng density score
        road_branch_left = self._has_side_exit(road, side="left")
        road_branch_right = self._has_side_exit(road, side="right")
        branch_left = bool(marking_topology['branch_left'])
        branch_right = bool(marking_topology['branch_right'])
        straight = bool(marking_topology['straight'] and front_ok)
        width_ok = width_frac >= self.width_min

        side_ok = branch_left or branch_right
        shape_ok = bool(marking_topology['horizontal_ok'] and side_ok)
        if self.require_front:
            raw_intersection = center_ok and front_ok and straight and shape_ok
        else:
            raw_intersection = center_ok and shape_ok

        if not center_ok:
            decision_reason = 'waiting: road center is not solid enough'
        elif not marking_topology['horizontal_ok']:
            decision_reason = 'waiting: no horizontal T/+ bar'
        elif self.require_stem_for_turn and not marking_topology['stem_down_ok']:
            decision_reason = 'waiting: no vertical stem under the bar'
        elif not side_ok:
            decision_reason = 'waiting: horizontal bar does not reach left/right enough'
        elif self.require_front and not straight:
            decision_reason = 'waiting: + crossing required, but top stem is missing'
        elif raw_intersection:
            decision_reason = 'candidate: topology is valid, waiting for stable frames'
        else:
            decision_reason = 'waiting: topology is not valid yet'

        if now < self.cooldown_until:
            self.stable_count = 0
            self.missing_count = 0
            is_intersection = False
            decision_reason = 'cooldown: recently handled an intersection'
        else:
            if raw_intersection:
                self.stable_count += 1
                self.missing_count = 0
            else:
                self.missing_count += 1
                if self.missing_count >= self.stable_frames_exit:
                    self.stable_count = 0

            is_intersection = self.stable_count >= self.stable_frames_enter
            if is_intersection:
                decision_reason = 'ready: intersection confirmed'

        return {
            "is_intersection": is_intersection,
            "raw_intersection": raw_intersection,
            "straight": straight,
            "branch_left": branch_left,
            "branch_right": branch_right,
            "front_ok": front_ok,
            "center_ok": center_ok,
            "width_ok": width_ok,
            "front_score": front_score,
            "center_score": center_score,
            "left_score": left_score,
            "right_score": right_score,
            "width_frac": width_frac,
            "shape_ok": shape_ok,
            "decision_reason": decision_reason,
            "shape_type": marking_topology['shape_type'],
            "road_branch_left": road_branch_left,
            "road_branch_right": road_branch_right,
            "marking_mask": marking_mask,
            "horizontal_mask": marking_topology['horizontal_mask'],
            "vertical_mask": marking_topology['vertical_mask'],
            "horizontal_ok": marking_topology['horizontal_ok'],
            "stem_down_ok": marking_topology['stem_down_ok'],
            "stem_up_ok": marking_topology['stem_up_ok'],
            "bar_y": marking_topology['bar_y'],
            "bar_run": marking_topology['bar_run'],
            "stem_x": marking_topology['stem_x'],
            "bar_span": marking_topology['bar_span'],
            "left_span": marking_topology['left_span'],
            "right_span": marking_topology['right_span'],
            "down_span": marking_topology['down_span'],
            "up_span": marking_topology['up_span'],
            "bar_score": marking_topology['bar_score'],
            "stable_count": self.stable_count,
            "missing_count": self.missing_count,
            "mask": mask,
            "road": road,
            "rects": self.rects,
            "width_band": self.width_band,
            "detector_type": "topology",
            # Giữ key cũ để code cũ không bị KeyError.
            "horizontal_len": marking_topology['bar_span'],
            "horizontal_span": marking_topology['bar_span'],
            "horizontal_segments": [] if marking_topology['bar_run'] is None else [marking_topology['bar_run']],
        }

    def _has_side_exit(self, road, side="left"):
        """
        Kiểm tra có cửa rẽ trái/phải thật hay không.
        Điều kiện:
        1. Road phải chạm gần mép trái/phải ảnh.
        2. Vùng chạm phải nằm ở khoảng giữa phía trước, không phải sát đáy xe.
        3. Phải có đủ số hàng liên tục có road, tránh nhiễu nhỏ.
        """
        h, w = road.shape[:2]

        # Chỉ xét vùng phía trước xe, không xét quá gần đáy ảnh
        y1 = int(h * 0.36)
        y2 = int(h * 0.66)

        if side == "left":
            x1 = 0
            x2 = int(w * 0.10)
        else:
            x1 = int(w * 0.90)
            x2 = w

        strip = road[y1:y2, x1:x2] > 0

        if strip.size == 0:
            return False

        # Mỗi hàng có bao nhiêu pixel road ở sát mép
        row_ratio = np.mean(strip, axis=1)

        # Hàng nào có road đủ mạnh
        active_rows = row_ratio > 0.18

        # Cần đủ nhiều hàng liên tục/có mặt
        active_ratio = np.mean(active_rows)

        return active_ratio >= 0.22
# Dùng chung tên intersection_detector để các hàm drive/debug bên dưới gọi thống nhất.
intersection_detector = TopologyIntersectionDetector()
print("Topology intersection detector ready")


In [ ]:
# ============================================================
# Visual debug cho intersection detector
# Chạy được lâu giống road following, nhưng KHÔNG điều khiển motor.
# ============================================================

import cv2
import numpy as np
import time
import ipywidgets as widgets
from IPython.display import display


def ensure_debug_widget(width=448, height=224):
    """Tạo debug_widget nếu kernel chưa có biến này."""
    global debug_widget, status_widget

    try:
        debug_widget
    except NameError:
        debug_widget = widgets.Image(format='jpeg', width=width, height=height)
        display(debug_widget)

    try:
        status_widget
    except NameError:
        status_widget = widgets.HTML(value="Debug status: idle")
        display(status_widget)

    return debug_widget


def _draw_rect_by_ratio(img, rect, color, label=None):
    h, w = img.shape[:2]
    x1r, y1r, x2r, y2r = rect
    x1 = int(w * x1r)
    y1 = int(h * y1r)
    x2 = int(w * x2r)
    y2 = int(h * y2r)

    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    if label is not None:
        cv2.putText(
            img, label, (x1 + 3, y1 + 14),
            cv2.FONT_HERSHEY_SIMPLEX, 0.36,
            color, 1, cv2.LINE_AA
        )


def _yn(value):
    return "OK" if bool(value) else "NO"


def _intersection_state_text(result):
    if result.get("is_intersection", False):
        state = "READY"
    elif result.get("raw_intersection", False):
        state = "CANDIDATE"
    else:
        state = "WAIT"

    return "STATE:{} stable:{} reason:{}".format(
        state,
        result.get("stable_count", 0),
        result.get("decision_reason", "")
    )


def _intersection_mark_text(result):
    return "MARK:{} hbar={} stem={} top={} bar={:.2f} down={:.2f} up={:.2f}".format(
        result.get("shape_type", "none"),
        _yn(result.get("horizontal_ok", False)),
        _yn(result.get("stem_down_ok", False)),
        _yn(result.get("stem_up_ok", False)),
        result.get("bar_span", 0.0),
        result.get("down_span", 0.0),
        result.get("up_span", 0.0),
    )


def _intersection_turn_text(result):
    return "TURN:left={} right={} through(+)={}".format(
        _yn(result.get("branch_left", False)),
        _yn(result.get("branch_right", False)),
        _yn(result.get("straight", False)),
    )


def print_intersection_result(result, prefix=""):
    if prefix:
        print(prefix)
    print(_intersection_state_text(result))
    print(_intersection_mark_text(result))
    print(_intersection_turn_text(result))
    print(
        "ROAD:left={:.2f} right={:.2f} front={:.2f} center={:.2f} width={:.2f}".format(
            result.get("left_score", 0.0),
            result.get("right_score", 0.0),
            result.get("front_score", 0.0),
            result.get("center_score", 0.0),
            result.get("width_frac", 0.0),
        )
    )


def draw_topology_intersection_debug(image, result, show_right="road"):
    """
    Vẽ giống road following:
    - Nửa trái: camera + road overlay + gates left/front/center/right.
    - Nửa phải: road connected component hoặc raw road mask.

    show_right:
        "road" -> chỉ vùng road nối với xe, nên dùng mặc định.
        "mask" -> raw dark-road mask.
    """
    h, w = image.shape[:2]
    debug = image.copy()

    road = result.get("road")
    mask = result.get("mask")
    marking_mask = result.get("marking_mask")
    horizontal_mask = result.get("horizontal_mask")
    vertical_mask = result.get("vertical_mask")
    rects = result.get("rects", {})

    # Overlay vùng road nối với xe.
    if road is not None:
        overlay = debug.copy()
        overlay[road > 0] = (0, 255, 0)
        debug = cv2.addWeighted(debug, 0.72, overlay, 0.28, 0)

    if marking_mask is not None:
        overlay = debug.copy()
        overlay[marking_mask > 0] = (255, 255, 255)
        debug = cv2.addWeighted(debug, 0.82, overlay, 0.18, 0)

    bar_y = result.get("bar_y")
    bar_run = result.get("bar_run")
    stem_x = result.get("stem_x")
    if bar_y is not None and bar_run is not None:
        cv2.line(debug, (int(bar_run[0]), int(bar_y)), (int(bar_run[1]), int(bar_y)), (0, 255, 255), 3)
        cv2.putText(debug, "hbar", (int(bar_run[0]), max(12, int(bar_y) - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.30, (0, 255, 255), 1, cv2.LINE_AA)
    if stem_x is not None and bar_y is not None:
        cv2.line(debug, (int(stem_x), int(h * 0.24)), (int(stem_x), int(h * 0.94)), (255, 0, 255), 1)
        cv2.putText(debug, "stem", (min(w - 38, int(stem_x) + 3), int(h * 0.90)), cv2.FONT_HERSHEY_SIMPLEX, 0.30, (255, 0, 255), 1, cv2.LINE_AA)

    # Trục giữa camera.
    cv2.line(debug, (w // 2, 0), (w // 2, h - 1), (255, 255, 0), 1)
    cv2.putText(debug, "cyan=camera center", (5, h - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.30, (255, 255, 0), 1, cv2.LINE_AA)

    # Band đo độ mở ngang.
    if "width_band" in result:
        y1 = int(h * result["width_band"][0])
        y2 = int(h * result["width_band"][1])
        cv2.rectangle(debug, (0, y1), (w - 1, y2), (255, 0, 255), 1)

    # Vẽ gates.
    for name, rect in rects.items():
        if name == "left":
            ok = result.get("branch_left", False)
            score = result.get("left_score", 0.0)
        elif name == "right":
            ok = result.get("branch_right", False)
            score = result.get("right_score", 0.0)
        elif name == "front":
            ok = result.get("front_ok", False)
            score = result.get("front_score", 0.0)
        else:
            ok = result.get("center_ok", False)
            score = result.get("center_score", 0.0)

        color = (0, 255, 0) if ok else (0, 0, 255)
        gate_label = "road-{}".format(name)
        _draw_rect_by_ratio(debug, rect, color, "{}:{:.2f}".format(gate_label, score))

    status_color = (0, 255, 0) if result.get("is_intersection", False) else (0, 255, 255)
    overlay_lines = [
        _intersection_state_text(result)[:62],
        _intersection_mark_text(result)[:62],
        _intersection_turn_text(result),
        "ROAD:L={:.2f} R={:.2f} F={:.2f} C={:.2f} W={:.2f}".format(
            result.get("left_score", 0.0),
            result.get("right_score", 0.0),
            result.get("front_score", 0.0),
            result.get("center_score", 0.0),
            result.get("width_frac", 0.0),
        ),
    ]

    for idx, text in enumerate(overlay_lines):
        color = status_color if idx == 0 else (255, 255, 255)
        cv2.putText(
            debug,
            text,
            (5, 18 + idx * 18),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.34,
            color,
            1,
            cv2.LINE_AA
        )

    # Right panel.
    if show_right == "marking" and marking_mask is not None:
        right_panel = marking_mask
    elif show_right == "horizontal" and horizontal_mask is not None:
        right_panel = horizontal_mask
    elif show_right == "vertical" and vertical_mask is not None:
        right_panel = vertical_mask
    elif show_right == "mask" and mask is not None:
        right_panel = mask
    elif road is not None:
        right_panel = road
    else:
        right_panel = np.zeros((h, w), dtype=np.uint8)

    right_panel_color = cv2.cvtColor(right_panel, cv2.COLOR_GRAY2BGR)
    combined = np.concatenate([debug, right_panel_color], axis=1)
    return combined


# Alias để lệnh cũ vẫn chạy được.
def make_intersection_debug_image(image, inter, show_right="road"):
    return draw_topology_intersection_debug(image, inter, show_right=show_right)


def live_topology_intersection_debug(duration_sec=30, interval_sec=0.08, show_right="road", print_every_sec=1.0):
    """
    Test detect ngã tư live trong nhiều giây.
    KHÔNG chạy motor.
    """
    ensure_debug_widget()

    try:
        car.throttle = 0.0
        car.steering = 0.0
    except Exception:
        pass

    intersection_detector.reset()

    start = time.time()
    last_print = 0.0
    frame_count = 0

    status_widget.value = "Debug status: topology intersection running"
    print("Start topology intersection debug. Motor disabled.")

    try:
        while time.time() - start < duration_sec:
            frame_count += 1
            image = camera.value
            result = intersection_detector.detect(image)

            debug_img = draw_topology_intersection_debug(image, result, show_right=show_right)
            debug_widget.value = bgr8_to_jpeg(debug_img)

            now = time.time()
            if now - last_print >= print_every_sec:
                last_print = now
                print_intersection_result(
                    result,
                    prefix="t: {:.1f}".format(now - start)
                )

            time.sleep(interval_sec)

    except KeyboardInterrupt:
        print("Interrupted")

    finally:
        try:
            car.throttle = 0.0
            car.steering = 0.0
        except Exception:
            pass

        status_widget.value = "Debug status: idle"
        print("Stop topology intersection debug. frames:", frame_count)


# Alias để lệnh bạn đã dùng vẫn chạy được.
def live_intersection_draw_debug(duration_sec=30, interval_sec=0.08, show_right="road"):
    return live_topology_intersection_debug(
        duration_sec=duration_sec,
        interval_sec=interval_sec,
        show_right=show_right
    )


# Test detect lâu giống road following:
# live_topology_intersection_debug(duration_sec=60)
# live_intersection_draw_debug(duration_sec=60)


In [ ]:
# ============================================================
# Intersection Drive State Machine
# FOLLOW -> APPROACH -> TURN_LEFT / TURN_RIGHT / STRAIGHT -> REALIGN -> FOLLOW
# Dùng TopologyIntersectionDetector ở cell trên.
# ============================================================

import math
import time


def clamp(x, lo, hi):
    return max(min(x, hi), lo)


def apply_steering_smoothing(target, previous, alpha=0.72, max_delta=0.16):
    smoothed = previous + alpha * (target - previous)
    delta = smoothed - previous
    delta = clamp(delta, -max_delta, max_delta)
    return previous + delta


def enforce_min_abs_steering(steering, min_abs, fallback_sign=1):
    if abs(steering) >= min_abs:
        return steering
    if steering > 0:
        return min_abs
    elif steering < 0:
        return -min_abs
    else:
        return min_abs if fallback_sign >= 0 else -min_abs


def compute_lane_control(image, result, last_mode, border_risk_frames, last_steering):
    """
    Road following control tách riêng để dùng lại trong FOLLOW và REALIGN.
    """
    STEERING_SIGN = -1

    K_angle_normal = 0.026
    K_lat_normal = 0.0035

    K_angle_curve = 0.060
    K_lat_curve = 0.0060

    K_angle_recovery = 0.085
    K_lat_recovery = 0.0090

    steering_bias = 0.0

    throttle_normal = 0.15
    throttle_curve = 0.12
    throttle_recovery = 0.10

    max_steering_normal = 0.50
    max_steering_curve = 0.85
    max_steering_recovery = 1.00

    min_recovery_steering = 0.65

    curve_angle_enter = 5.0
    curve_angle_exit = 3.0

    recovery_lat_enter = 35.0
    recovery_lat_exit = 22.0

    recovery_angle_enter = 18.0
    recovery_angle_exit = 10.0

    steering_alpha = 0.72

    max_steering_delta_normal = 0.11
    max_steering_delta_curve = 0.18
    max_steering_delta_recovery = 0.25

    max_border_risk_frames = 10

    geom = get_road_geometry(result, detector, image)

    angle_error = geom["angle_deg"]
    lateral_error = geom["lateral_error"]
    border_risk = geom.get("border_risk", False)

    if border_risk:
        border_risk_frames += 1
    else:
        border_risk_frames = 0

    abs_angle = abs(angle_error)
    abs_lat = abs(lateral_error)
    force_recovery = border_risk_frames >= max_border_risk_frames

    if force_recovery:
        mode = "RECOVERY"
    elif last_mode == "RECOVERY":
        if abs_lat <= recovery_lat_exit and abs_angle <= recovery_angle_exit:
            if abs_angle >= curve_angle_enter:
                mode = "CURVE"
            else:
                mode = "NORMAL"
        else:
            mode = "RECOVERY"
    elif last_mode == "CURVE":
        if abs_lat >= recovery_lat_enter or abs_angle >= recovery_angle_enter:
            mode = "RECOVERY"
        elif abs_angle <= curve_angle_exit:
            mode = "NORMAL"
        else:
            mode = "CURVE"
    else:
        if abs_lat >= recovery_lat_enter or abs_angle >= recovery_angle_enter:
            mode = "RECOVERY"
        elif abs_angle >= curve_angle_enter or border_risk:
            mode = "CURVE"
        else:
            mode = "NORMAL"

    if mode == "RECOVERY":
        K_angle = K_angle_recovery
        K_lat = K_lat_recovery
        throttle = throttle_recovery
        max_steering = max_steering_recovery
        max_delta = max_steering_delta_recovery
    elif mode == "CURVE":
        K_angle = K_angle_curve
        K_lat = K_lat_curve
        throttle = throttle_curve
        max_steering = max_steering_curve
        max_delta = max_steering_delta_curve
    else:
        K_angle = K_angle_normal
        K_lat = K_lat_normal
        throttle = throttle_normal
        max_steering = max_steering_normal
        max_delta = max_steering_delta_normal

    raw_control = K_angle * angle_error + K_lat * lateral_error
    target_steering = STEERING_SIGN * raw_control + steering_bias
    target_steering = clamp(target_steering, -max_steering, max_steering)

    if mode == "RECOVERY":
        fallback_sign = 1 if target_steering >= 0 else -1
        target_steering = enforce_min_abs_steering(
            target_steering,
            min_recovery_steering,
            fallback_sign=fallback_sign
        )
        target_steering = clamp(target_steering, -max_steering, max_steering)

    steering = apply_steering_smoothing(
        target=target_steering,
        previous=last_steering,
        alpha=steering_alpha,
        max_delta=max_delta
    )
    steering = clamp(steering, -max_steering, max_steering)

    return {
        "steering": steering,
        "throttle": throttle,
        "mode": mode,
        "border_risk_frames": border_risk_frames,
        "geom": geom,
        "raw_control": raw_control,
        "target_steering": target_steering
    }


def run_intersection_drive(turn_direction="right", duration_sec=40, turn_right_sec=None, turn_left_sec=None):
    """
    turn_direction:
        "left"     -> gặp ngã tư thì rẽ trái
        "right"    -> gặp ngã tư thì rẽ phải
        "straight" -> gặp ngã tư thì đi thẳng

    Sau khi rẽ xong, state REALIGN dùng lại road following để bắt lane mới.
    """
    assert turn_direction in ["left", "right", "straight"]

    # Nếu xe rẽ ngược, đổi dấu 2 dòng dưới.
    TURN_LEFT_STEERING = -0.85
    TURN_RIGHT_STEERING = 0.95

    TURN_THROTTLE = 0.12
    APPROACH_THROTTLE = 0.11
    REALIGN_THROTTLE_MAX = 0.10
    LOST_THROTTLE = 0.020

    APPROACH_SEC = 0.25
    TURN_LEFT_SEC = 1.05 if turn_left_sec is None else turn_left_sec
    TURN_RIGHT_SEC = 1.30 if turn_right_sec is None else turn_right_sec
    STRAIGHT_CROSS_SEC = 0.70

    REALIGN_TIMEOUT_SEC = 2.5
    REALIGN_STABLE_FRAMES = 6

    print_every_n_frames = 10
    debug_every_n_frames = 2

    state = "FOLLOW"
    state_start = time.time()

    last_steering = 0.0
    last_mode = "NORMAL"
    border_risk_frames = 0
    lost_frames = 0
    max_lost_frames = 4
    relock_frames = 0

    try:
        detector.error_history.clear()
    except Exception:
        pass

    ensure_debug_widget()
    intersection_detector.suppress()

    car.throttle = 0.0
    car.steering = 0.0

    print("Starting in 2 seconds...")
    time.sleep(2)

    start = time.time()
    frame_id = 0
    inter = None

    try:
        while time.time() - start < duration_sec:
            frame_id += 1
            now = time.time()
            image = camera.value

            # ============================================================
            # STATE: APPROACH / TURN / STRAIGHT
            # ============================================================
            if state == "APPROACH":
                car.steering = 0.0
                car.throttle = APPROACH_THROTTLE

                if now - state_start >= APPROACH_SEC:
                    if turn_direction == "left":
                        state = "TURN_LEFT"
                    elif turn_direction == "right":
                        state = "TURN_RIGHT"
                    else:
                        state = "STRAIGHT"

                    state_start = now
                    if state == "TURN_RIGHT":
                        print("STATE -> TURN_RIGHT", "sec:", TURN_RIGHT_SEC, "steer:", TURN_RIGHT_STEERING)
                    elif state == "TURN_LEFT":
                        print("STATE -> TURN_LEFT", "sec:", TURN_LEFT_SEC, "steer:", TURN_LEFT_STEERING)
                    else:
                        print("STATE ->", state, "sec:", STRAIGHT_CROSS_SEC)

                time.sleep(0.04)
                continue

            if state == "TURN_LEFT":
                car.steering = TURN_LEFT_STEERING
                car.throttle = TURN_THROTTLE

                if now - state_start >= TURN_LEFT_SEC:
                    state = "REALIGN"
                    state_start = now
                    relock_frames = 0
                    print("STATE -> REALIGN")

                time.sleep(0.04)
                continue

            if state == "TURN_RIGHT":
                car.steering = TURN_RIGHT_STEERING
                car.throttle = TURN_THROTTLE

                if now - state_start >= TURN_RIGHT_SEC:
                    state = "REALIGN"
                    state_start = now
                    relock_frames = 0
                    print("STATE -> REALIGN")

                time.sleep(0.04)
                continue

            if state == "STRAIGHT":
                car.steering = 0.0
                car.throttle = APPROACH_THROTTLE

                if now - state_start >= STRAIGHT_CROSS_SEC:
                    state = "REALIGN"
                    state_start = now
                    relock_frames = 0
                    print("STATE -> REALIGN")

                time.sleep(0.04)
                continue

            # ============================================================
            # FOLLOW / REALIGN: detect lane
            # ============================================================
            result = detector.detect(image)

            if (not result["ok"]) or result.get("lane_center") is None:
                lost_frames += 1

                if state == "REALIGN":
                    car.steering = clamp(last_steering, -0.75, 0.75)
                    car.throttle = LOST_THROTTLE

                    if now - state_start > REALIGN_TIMEOUT_SEC:
                        car.throttle = 0.0
                        car.steering = 0.0
                        print("STOP: cannot relock lane after turn")
                        break

                    time.sleep(0.04)
                    continue

                if lost_frames <= max_lost_frames:
                    car.steering = clamp(last_steering, -0.75, 0.75)
                    car.throttle = LOST_THROTTLE
                    time.sleep(0.04)
                    continue

                car.throttle = 0.0
                car.steering = 0.0
                print("STOP: lane lost too long")
                break

            lost_frames = 0

            # ============================================================
            # FOLLOW: detect intersection bằng topology
            # ============================================================
            inter = intersection_detector.detect(image)

            if state == "FOLLOW":
                wanted_available = (
                    (turn_direction == "left" and inter["branch_left"])
                    or (turn_direction == "right" and inter["branch_right"])
                    or (turn_direction == "straight" and inter["straight"])
                )

                if inter["is_intersection"] and wanted_available:
                    car.throttle = 0.0
                    car.steering = 0.0

                    intersection_detector.suppress()
                    state = "APPROACH"
                    state_start = now

                    print_intersection_result(
                        inter,
                        prefix="INTERSECTION FOUND -> APPROACH dir={}".format(turn_direction)
                    )

                    time.sleep(0.05)
                    continue

            # ============================================================
            # Lane control
            # ============================================================
            control = compute_lane_control(
                image=image,
                result=result,
                last_mode=last_mode,
                border_risk_frames=border_risk_frames,
                last_steering=last_steering
            )

            steering = control["steering"]
            throttle = control["throttle"]
            last_mode = control["mode"]
            border_risk_frames = control["border_risk_frames"]

            if state == "REALIGN":
                throttle = min(throttle, REALIGN_THROTTLE_MAX)

                geom = control["geom"]
                if abs(geom["lateral_error"]) < 24 and abs(geom["angle_deg"]) < 12:
                    relock_frames += 1
                else:
                    relock_frames = 0

                if relock_frames >= REALIGN_STABLE_FRAMES:
                    state = "FOLLOW"
                    state_start = now
                    last_mode = "NORMAL"
                    intersection_detector.suppress()
                    print("STATE -> FOLLOW")

                if now - state_start > REALIGN_TIMEOUT_SEC:
                    car.throttle = 0.0
                    car.steering = 0.0
                    print("STOP: realign timeout")
                    break

            car.steering = steering
            car.throttle = throttle
            last_steering = steering

            # ============================================================
            # Debug
            # ============================================================
            if frame_id % debug_every_n_frames == 0:
                try:
                    debug_img = draw_topology_intersection_debug(image, inter, show_right="marking")
                    debug_widget.value = bgr8_to_jpeg(debug_img)
                except Exception:
                    pass

            if frame_id % print_every_n_frames == 0:
                geom = control["geom"]
                print(
                    "DRIVE: state={} mode={} angle={:.2f} lat={:.2f} steer={:.3f} thr={:.3f}".format(
                        state,
                        last_mode,
                        geom["angle_deg"],
                        geom["lateral_error"],
                        steering,
                        throttle,
                    )
                )
                print_intersection_result(inter)

            time.sleep(0.04)

    except KeyboardInterrupt:
        print("Interrupted")

    finally:
        car.throttle = 0.0
        car.steering = 0.0
        status_widget.value = "Debug status: idle"
        print("STOPPED")


# Chạy khi đã test detect ổn:
# run_intersection_drive(turn_direction="right", duration_sec=40)
# run_intersection_drive(turn_direction="left", duration_sec=40)
# run_intersection_drive(turn_direction="straight", duration_sec=40)


In [ ]:
# ============================================================
# Scenario tests: Canny lane following + topology intersection
# ============================================================


def _reset_lane_detector_state():
    try:
        detector.error_history.clear()
    except Exception:
        pass

    try:
        detector.last_center = None
        detector.lost_count = 0
    except Exception:
        pass


def _set_target_lane(target_lane):
    assert target_lane in ["RIGHT_LANE", "LEFT_LANE"]
    detector.target_lane = target_lane
    _reset_lane_detector_state()
    print("TARGET_LANE ->", detector.target_lane)


def _get_target_lane():
    return getattr(detector, "target_lane", "RIGHT_LANE")


def _restore_target_lane(target_lane):
    detector.target_lane = target_lane
    _reset_lane_detector_state()
    print("TARGET_LANE restored ->", detector.target_lane)


def _stop_car():
    try:
        car.throttle = 0.0
        car.steering = 0.0
    except Exception:
        pass


def apply_lane_camera_profile(profile="ORIGINAL_CAMERA"):
    """Apply lane-following calibration for the current camera angle."""
    profile = profile.upper()

    if profile in ["ORIGINAL", "ORIGINAL_CAMERA", "LOW_CAMERA"]:
        detector.roi_start_ratio = 0.35
        detector.poly_top_left = 35
        detector.poly_top_right = 190
        detector.poly_bottom_left = 5
        detector.poly_bottom_right = 219
        detector.lookahead_ratio = 0.75
        detector.geom_far_ratio = 0.35
        detector.geom_near_ratio = 0.82
        detector.target_offset_px = -18
        detector.right_lane_bias_px = 0
        detector.left_lane_bias_px = 0
        detector.single_line_extra_offset_px = 30
        detector.min_right_clearance_px = 48
        detector.min_left_clearance_px = 34
        detector.lane_width_pixels = 90

    elif profile in ["RAISED", "RAISED_CAMERA", "HIGH_CAMERA"]:
        detector.roi_start_ratio = 0.45
        detector.poly_top_left = 48
        detector.poly_top_right = 176
        detector.poly_bottom_left = 8
        detector.poly_bottom_right = 216
        detector.lookahead_ratio = 0.82
        detector.geom_far_ratio = 0.48
        detector.geom_near_ratio = 0.90
        detector.target_offset_px = 0
        detector.right_lane_bias_px = 10
        detector.left_lane_bias_px = -10
        detector.single_line_extra_offset_px = 10
        detector.min_right_clearance_px = 36
        detector.min_left_clearance_px = 28
        detector.lane_width_pixels = 75

    else:
        raise ValueError("Unknown profile: {}".format(profile))

    try:
        detector.error_history.clear()
    except Exception:
        pass
    try:
        detector.last_center = None
        detector.lost_count = 0
    except Exception:
        pass

    print("LANE_CAMERA_PROFILE ->", profile)
    print(
        "roi={:.2f} lookahead={:.2f} lane_width={} target_offset={} bias_R={} bias_L={} one_line_extra={}".format(
            detector.roi_start_ratio,
            detector.lookahead_ratio,
            detector.lane_width_pixels,
            detector.target_offset_px,
            detector.right_lane_bias_px,
            detector.left_lane_bias_px,
            detector.single_line_extra_offset_px,
        )
    )


def reset_lane_following_original():
    """Restore the original low-camera CannyEdgeRoadFollowing calibration."""
    apply_lane_camera_profile("ORIGINAL_CAMERA")
    detector.target_lane = "RIGHT_LANE"
    print("Lane following restored to original Canny default: RIGHT_LANE")


def reset_lane_following_raised():
    """Use this after raising/tilting the camera upward."""
    apply_lane_camera_profile("RAISED_CAMERA")
    intersection_detector.center_min = 0.14
    intersection_detector.stable_frames_enter = 2
    intersection_detector.mark_search_y = (0.28, 0.80)
    intersection_detector.horizontal_min_span = 0.28
    intersection_detector.branch_min_span = 0.10
    detector.target_lane = "RIGHT_LANE"
    print("Lane following calibrated for raised camera: RIGHT_LANE")


def run_lane_following_no_turn(duration_sec=40, target_lane=None, straight_hold_sec=0.85):
    """Lane-following only. It ignores intersections and never starts a turn."""
    original_target_lane = _get_target_lane()
    if target_lane is None:
        try:
            detector.error_history.clear()
        except Exception:
            pass
        print("TARGET_LANE kept ->", detector.target_lane)
    else:
        _set_target_lane(target_lane)
    ensure_debug_widget()

    _stop_car()
    print("LANE-ONLY TEST: no intersection turn will be executed")
    print("Starting in 2 seconds...")
    time.sleep(2)

    last_steering = 0.0
    last_mode = "NORMAL"
    border_risk_frames = 0
    lost_frames = 0
    max_lost_frames = 4
    lost_throttle = 0.020
    frame_id = 0
    debug_every_n_frames = 2
    print_every_n_frames = 10
    straight_hold_until = 0.0
    start = time.time()

    try:
        while time.time() - start < duration_sec:
            frame_id += 1
            now = time.time()
            image = camera.value
            result = detector.detect(image)

            try:
                inter = intersection_detector.detect(image)
            except Exception:
                inter = {"is_intersection": False, "raw_intersection": False}

            intersection_seen = inter.get("is_intersection", False) or inter.get("raw_intersection", False)
            if intersection_seen and now >= straight_hold_until:
                straight_hold_until = now + straight_hold_sec
                try:
                    intersection_detector.suppress()
                except Exception:
                    pass
                print_intersection_result(
                    inter,
                    prefix="LANE_ONLY: intersection seen -> STRAIGHT_HOLD {:.2f}s".format(straight_hold_sec)
                )

            if now < straight_hold_until:
                car.steering = 0.0
                car.throttle = 0.11
                last_steering = 0.0
                last_mode = "STRAIGHT_HOLD"

                if frame_id % debug_every_n_frames == 0:
                    try:
                        debug_widget.value = bgr8_to_jpeg(make_debug_image(image, result))
                    except Exception:
                        pass

                if frame_id % print_every_n_frames == 0:
                    print("LANE_ONLY STRAIGHT_HOLD steering=0.000 thr=0.110")

                time.sleep(0.04)
                continue

            if (not result["ok"]) or result.get("lane_center") is None:
                lost_frames += 1

                if lost_frames <= max_lost_frames:
                    safe_steering = clamp(last_steering, -0.75, 0.75)
                    car.steering = safe_steering
                    car.throttle = lost_throttle

                    if frame_id % print_every_n_frames == 0:
                        print(
                            "LANE_ONLY LOST_RECOVERY",
                            "lost:", lost_frames,
                            "steer:", round(safe_steering, 3),
                            "thr:", lost_throttle
                        )

                    time.sleep(0.04)
                    continue

                _stop_car()
                print("STOP: lane lost too long")
                break

            lost_frames = 0
            control = compute_lane_control(
                image=image,
                result=result,
                last_mode=last_mode,
                border_risk_frames=border_risk_frames,
                last_steering=last_steering
            )

            steering = control["steering"]
            throttle = control["throttle"]
            last_mode = control["mode"]
            border_risk_frames = control["border_risk_frames"]

            car.steering = steering
            car.throttle = throttle
            last_steering = steering

            if frame_id % debug_every_n_frames == 0:
                try:
                    debug_widget.value = bgr8_to_jpeg(make_debug_image(image, result))
                except Exception:
                    pass

            if frame_id % print_every_n_frames == 0:
                geom = control["geom"]
                print(
                    "LANE_ONLY lane={} mode={} angle={:.2f} lat={:.2f} steer={:.3f} thr={:.3f} left={} right={} width={} offset={} bias_R={} bias_L={}".format(
                        detector.target_lane,
                        last_mode,
                        geom["angle_deg"],
                        geom["lateral_error"],
                        steering,
                        throttle,
                        result.get("left_x"),
                        result.get("right_x"),
                        detector.lane_width_pixels,
                        detector.target_offset_px,
                        getattr(detector, "right_lane_bias_px", 0),
                        getattr(detector, "left_lane_bias_px", 0),
                    )
                )

            time.sleep(0.04)

    except KeyboardInterrupt:
        print("Interrupted")

    finally:
        _stop_car()
        try:
            status_widget.value = "Debug status: idle"
        except Exception:
            pass
        if target_lane is not None:
            _restore_target_lane(original_target_lane)
        print("STOPPED")


def test_1_straight_no_turn(duration_sec=40, target_lane=None, straight_hold_sec=0.85):
    """Test 1: go straight/follow lane only, never turn at an intersection."""
    return run_lane_following_no_turn(
        duration_sec=duration_sec,
        target_lane=target_lane,
        straight_hold_sec=straight_hold_sec
    )


def test_2_right_turn_same_lane(duration_sec=40, turn_right_sec=1.30):
    """Test 2: start in RIGHT_LANE and turn right when topology allows it."""
    original_target_lane = _get_target_lane()
    print("Place the car in RIGHT_LANE before starting this test.")
    try:
        _set_target_lane("RIGHT_LANE")
        return run_intersection_drive(
            turn_direction="right",
            duration_sec=duration_sec,
            turn_right_sec=turn_right_sec
        )
    finally:
        _restore_target_lane(original_target_lane)


def test_3_left_turn_other_lane(duration_sec=40, turn_left_sec=1.05):
    """Test 3: start in LEFT_LANE and turn left when topology allows it."""
    original_target_lane = _get_target_lane()
    print("Place the car in LEFT_LANE before starting this test.")
    print("If the car is physically in RIGHT_LANE, this test will try to acquire LEFT_LANE and may drive onto the divider line.")
    try:
        _set_target_lane("LEFT_LANE")
        return run_intersection_drive(
            turn_direction="left",
            duration_sec=duration_sec,
            turn_left_sec=turn_left_sec
        )
    finally:
        _restore_target_lane(original_target_lane)


# Short aliases.
test_straight = test_1_straight_no_turn
test_right_same_lane = test_2_right_turn_same_lane
test_left_other_lane = test_3_left_turn_other_lane

reset_lane_following_raised()

print("Scenario tests ready:")
print("Camera profile default: RAISED_CAMERA")
print("1) test_straight(duration_sec=40)  # keeps current lane target")
print("2) test_right_same_lane(duration_sec=40)")
print("3) test_left_other_lane(duration_sec=40)")


In [ ]:
# Examples:
# live_topology_intersection_debug(duration_sec=60, show_right="marking")
# test_straight(duration_sec=40)
# test_right_same_lane(duration_sec=40)
# test_left_other_lane(duration_sec=40)

## Cách chạy nhanh

1. Chạy từ trên xuống đến cell tạo `car`.
2. Chạy cell `Topology Intersection Detector` và cell `Scenario tests`.
3. Nếu camera đã nâng/ngẩng lên, giữ profile mặc định mới hoặc gọi lại:

```python
reset_lane_following_raised()
```

Nếu hạ camera về góc cũ của file Canny gốc:

```python
reset_lane_following_original()
```

4. Test detect trước, không chạy motor:

```python
live_topology_intersection_debug(duration_sec=60, show_right="marking")
```

Hoặc lệnh cũ vẫn dùng được:

```python
live_intersection_draw_debug(duration_sec=60)
```

5. Chạy một trong 3 scenario test:

```python
# Test 1: đi thẳng / follow lane, không rẽ bên nào cả.
test_straight(duration_sec=40)

# Test 2: đang đi RIGHT_LANE và rẽ phải cùng lane.
test_right_same_lane(duration_sec=40)

# Test 3: đang đi LEFT_LANE và rẽ trái khác lane.
test_left_other_lane(duration_sec=40)
```

Lưu ý: `test_straight()` giữ nguyên `detector.target_lane` hiện tại giống lane-following gốc. `test_left_other_lane()` chỉ nên chạy khi xe thật sự đang đặt ở `LEFT_LANE`; nếu xe đang ở `RIGHT_LANE`, nó sẽ cố bắt lane trái và có thể chạy lên vạch chia lane.

Nếu `test_straight()` tới ngã tư vẫn có xu hướng rẽ phải, tăng thời gian giữ thẳng:

```python
test_straight(duration_sec=40, straight_hold_sec=1.1)
```

Nếu xe ở `RIGHT_LANE` còn áp vào vạch đứt bên trái, tăng bias sang phải:

```python
detector.right_lane_bias_px = 14
test_straight(duration_sec=40)
```

Nếu rẽ phải chưa đủ sâu, tăng thời gian turn:

```python
test_right_same_lane(duration_sec=40, turn_right_sec=1.45)
```

Nếu rẽ phải quá sâu, giảm về khoảng `1.15` đến `1.25`.


Nếu đang đi đường cong mà vẫn báo `IX: True`, tăng ngưỡng topology vạch trắng:

```python
intersection_detector.horizontal_min_span = 0.40
intersection_detector.branch_min_span = 0.18
intersection_detector.vertical_min_span = 0.20
```

Nếu tới ngã tư/ngã ba thật mà không detect, nhìn `shape/bar/down/up` trong log rồi hạ ngưỡng tương ứng:

```python
intersection_detector.horizontal_min_span = 0.30
intersection_detector.branch_min_span = 0.12
intersection_detector.vertical_min_span = 0.12
```

Nếu chỉ muốn bắt ngã tư dấu `+` và bỏ qua ngã ba chữ `T`:

```python
intersection_detector.require_front = True
```

## How to read the debug log

- `STATE:WAIT` means the detector is not ready to turn yet.
- `STATE:CANDIDATE` means the current frame looks like an intersection, but it is waiting for stable frames.
- `STATE:READY` means the intersection is confirmed.
- `MARK:t_junction` means it sees a horizontal `T` marking, usually a T-junction.
- `MARK:plus` means it sees a `+` marking, usually a 4-way intersection.
- `hbar=OK` means the horizontal bar exists.
- `stem=OK` means the vertical stem under the bar exists. It is useful diagnostic info, but left/right turns are allowed from `hbar=OK` by default.
- Set `intersection_detector.require_stem_for_turn = True` if you want the stricter old behavior.
- `top=OK` means the upper stem exists, so `through(+)` is allowed.
- `TURN:left/right/through(+)` tells which directions are currently allowed.
- `through(+)` means crossing a 4-way `+` intersection, not simply seeing road in front.
- The green/red boxes are road-mask gates: `road-left`, `road-right`, `road-front`, `road-center`.
- Gate boxes only check whether dark road exists in those regions; they are secondary diagnostics.
- The main turn decision comes from `MARK` and `TURN`, especially the yellow horizontal bar.
- Cyan vertical line is the camera center. Magenta vertical line is the detected marking/stem center. They do not have to perfectly overlap.
